# Домашнее задание блока 1. Лекции 1–2

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/homeworks/hw1/homework.ipynb)

Домашние задания в курсе выдаются **раз в две недели** и охватывают обе лекции блока. Это задание — по лекциям 1 и 2: первый алгоритм (Cross-Entropy) и оценка политики по Монте-Карло, своя среда в Gymnasium, точная модель MDP, уравнения Беллмана и оптимальная политика.

Всего **100 баллов** плюс до 30 бонусных:

1. **Метод Cross-Entropy на Frozen Lake 8×8 — 25 баллов**
2. **Оценка политики по Монте-Карло — 10 баллов**
3. **Своя среда: управление запасами — 20 баллов**
4. **Уравнения Беллмана на складе — 25 баллов**
5. **Теория — 20 баллов**
6. **Бонус А: скрытый день недели — 10 баллов**
7. **Бонус Б: многорукие бандиты — 10 баллов**
8. **Бонус В: DL-разминка — 10 баллов** (для тех, кто прошёл `../../01-intro/seminar/dl_basics.ipynb`)

Части с `assert` проверяются автоматически при запуске ячейки: если assert не упал, часть засчитана. Текстовые ответы и графики проверяются вручную — пишите их прямо в markdown-ячейках вместо слова `TODO`.

Весь ноутбук держится на одном наборе функций: `run_session`, `discounted_return`, `evaluate` и `cross_entropy_method`. Их вы пишете в частях 1–2 и дальше переиспользуете на складе, поэтому не переопределяйте их по ходу ноутбука. Политика везде — таблица «состояние × действие» с вероятностями.

**Не забудьте про квизы.** К этому блоку их два — по лекции 1 и по лекции 2. Пройдите оба в зачётном режиме и положите коды результатов в `quiz/week01.txt` и `quiz/week02.txt`.

Первые ячейки служебные: установка пакетов в Colab и функции для картинок из лекции. Запустите и идите дальше.

In [ ]:
# Если ноутбук открыт в Google Colab: ставим недостающие пакеты. Локально (после uv sync) ячейка ничего не делает.
import importlib.util, subprocess, sys
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium[toy-text,classic-control]"], check=True)

In [ ]:
# Служебный код для картинок: запустить и не читать. Содержательный код начинается со следующей ячейки.
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from matplotlib import animation
from IPython.display import HTML, display

ARROWS = "←↓→↑"                      # так пронумерованы действия в Frozen Lake: 0=←, 1=↓, 2=→, 3=↑
CELL_COLORS = {"S": "#f2f2f2", "F": "#ffffff", "H": "#a8c8ec", "G": "#a9dfa9"}

def lake_desc(env):
    """Карта озера как список строк: ['SFFF', 'FHFH', 'FFFH', 'HFFG']."""
    return ["".join(c.decode() for c in row) for row in env.unwrapped.desc]

def draw_lake(desc, ax=None, size=3.0):
    """Пустая карта озера. Возвращает оси, поверх которых можно рисовать маршруты и стрелки."""
    nrow, ncol = len(desc), len(desc[0])
    if ax is None:
        _, ax = plt.subplots(figsize=(size, size * nrow / ncol))
    ax.set_xlim(0, ncol); ax.set_ylim(nrow, 0); ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
    for r in range(nrow):
        for c in range(ncol):
            ax.add_patch(plt.Rectangle((c, r), 1, 1, facecolor=CELL_COLORS[desc[r][c]], edgecolor="k", lw=0.6))
            if desc[r][c] in "SHG":
                ax.text(c + 0.5, r + 0.82, desc[r][c], ha="center", va="center", fontsize=8, color="#666")
    return ax

def draw_paths(sessions, desc, ax=None, title="", color="C0", alpha=0.35, size=3.0):
    """Маршруты эпизодов поверх карты: ломаная через центры клеток, точка — где эпизод закончился."""
    ax = draw_lake(desc, ax, size)
    ncol, rng = len(desc[0]), np.random.default_rng(0)
    for states, *_ in sessions:
        pts = np.array([divmod(int(s), ncol) for s in states], float) + 0.5 + rng.normal(0, 0.07, (len(states), 2))
        ax.plot(pts[:, 1], pts[:, 0], color=color, alpha=alpha, lw=1.5)
        ax.plot(pts[-1, 1], pts[-1, 0], "o", color=color, alpha=alpha, ms=4)
    ax.set_title(title, fontsize=10)
    return ax

def draw_policy(policy, desc, ax=None, title="", size=3.0):
    """Стрелка — самое вероятное действие, насыщенность — его вероятность; «·» — строка почти равномерная."""
    ax = draw_lake(desc, ax, size)
    ncol = len(desc[0])
    for s, p in enumerate(policy):
        r, c = divmod(s, ncol)
        if desc[r][c] in "HG":
            continue
        a, top2 = int(np.argmax(p)), np.sort(p)[-2:]
        if top2[1] - top2[0] < 0.05:
            ax.text(c + 0.5, r + 0.5, "·", ha="center", va="center", fontsize=18, color="grey")
        else:
            ax.text(c + 0.5, r + 0.5, ARROWS[a], ha="center", va="center", fontsize=20, alpha=0.2 + 0.8 * float(p[a]))
    ax.set_title(title, fontsize=10)
    return ax

def plot_values(values, desc, ax=None, title="", vmax=None, size=3.0):
    """Тепловая карта ценности клеток (проруби и цель — терминальные, у них ценность 0 по определению)."""
    nrow, ncol = len(desc), len(desc[0])
    ax = draw_lake(desc, ax, size)
    grid = np.asarray(values, float).reshape(nrow, ncol)
    vmax = vmax or max(float(np.nanmax(grid)), 1e-9)
    for r in range(nrow):
        for c in range(ncol):
            if desc[r][c] in "HG":
                continue
            if np.isnan(grid[r, c]):                     # клетку ни разу не посещали — оценки нет
                ax.text(c + 0.5, r + 0.5, "—", ha="center", va="center", fontsize=9, color="grey")
                continue
            ax.add_patch(plt.Rectangle((c, r), 1, 1, facecolor=plt.cm.YlOrRd(0.85 * grid[r, c] / vmax), edgecolor="k", lw=0.6))
            ax.text(c + 0.5, r + 0.5, f"{grid[r, c]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title(title, fontsize=10)
    return ax

def plot_returns(returns, threshold=None, elite_mask=None, ax=None, title=""):
    """Каждая точка — эпизод, по вертикали его return (как на схеме метода Cross-Entropy)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.6, 3.2))
    returns, x = np.asarray(returns, float), np.random.default_rng(0).uniform(0, 1, len(returns))
    if elite_mask is None:
        ax.scatter(x, returns, s=22, color="C0", alpha=0.7)
    else:
        m = np.asarray(elite_mask, bool)
        ax.scatter(x[~m], returns[~m], s=18, color="grey", alpha=0.35, label=f"остальные ({(~m).sum()})")
        ax.scatter(x[m], returns[m], s=42, color="C2", alpha=0.95, edgecolors="white", label=f"элита ({m.sum()})")
        ax.legend(loc="upper right", fontsize=8)
    if threshold is not None:
        ax.axhline(threshold, ls="--", color="C3", lw=1.5)
        ax.text(0.0, threshold + 0.02, f"порог = {threshold:.2f}", color="C3", fontsize=8)
    ax.set_xticks([]); ax.set_ylabel("return"); ax.set_ylim(-0.05, 1.05); ax.set_title(title, fontsize=10)
    return ax

def animate_policy(policies, rates, desc, interval=500):
    """Кадр — одна итерация обучения: стрелки политики и доля успешных эпизодов."""
    nrow, ncol = len(desc), len(desc[0])
    fig, ax = plt.subplots(figsize=(3.4, 3.4 * nrow / ncol + 0.3)); plt.close(fig)
    def frame(k):
        ax.clear(); draw_policy(policies[k], desc, ax=ax, title=f"итерация {k}: доля успехов {rates[k]:.0%}")
    anim = animation.FuncAnimation(fig, frame, frames=len(policies), interval=interval)
    return HTML(anim.to_jshtml(default_mode="loop"))

def show_frames(frames, title="", interval=60, width=3.2):
    """Кадры среды -> анимация в ноутбуке (в превью на GitHub не видна, только при запуске)."""
    h, w = frames[0].shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h / w))
    ax.axis("off"); ax.set_title(title, fontsize=10)
    im = ax.imshow(frames[0]); plt.close(fig)
    anim = animation.FuncAnimation(fig, lambda i: (im.set_data(frames[i]),), frames=len(frames), interval=interval, blit=True)
    return HTML(anim.to_jshtml(default_mode="loop"))

Дальше — функции из лекции и семинара недели 1: `discounted_return`, `run_session` и оценка доли успехов. В частях 1–2 работаем с озером 8×8.

In [ ]:
GAMMA = 0.95

def discounted_return(rewards, gamma=GAMMA):
    """G = r_0 + γ r_1 + γ² r_2 + ..."""
    return sum(gamma ** t * r for t, r in enumerate(rewards))

def run_session(env, policy, rng, max_steps=100):
    """Один эпизод политикой-таблицей policy. Возвращает состояния (включая последнее), действия и награды."""
    s, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, rewards = [s], [], []
    for _ in range(max_steps):
        a = int(rng.choice(len(policy[s]), p=policy[s]))      # действие ~ π(· | s)
        s, r, terminated, truncated, _ = env.step(a)
        states.append(s); actions.append(a); rewards.append(r)
        if terminated or truncated:
            break
    return states, actions, rewards


def success_rate(env, policy, n_episodes=500, seed=0):
    """Доля эпизодов, дошедших до цели."""
    rng = np.random.default_rng(seed)
    return np.mean([sum(run_session(env, policy, rng)[2]) > 0 for _ in range(n_episodes)])

env8 = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)
desc8 = lake_desc(env8)
n_states, n_actions = env8.observation_space.n, env8.action_space.n
draw_lake(desc8, size=4); plt.show()
print("\n".join(desc8))

## Часть 1. Метод Cross-Entropy на Frozen Lake 8×8 (25 баллов)

На лекции и семинаре мы обучили табличный Cross-Entropy на озере 4×4. Карта 8×8 сложнее: 64 состояния, до цели далеко, случайная политика доходит реже, чем в одном эпизоде из пятисот.

### 1.1 Реализация и обучение (12 баллов)

Соберите алгоритм из трёх функций семинара: `select_elite(sessions, scores, q)`, `update_policy(policy, elite, laplace, mix)` и цикл `cross_entropy_method(...)`, возвращающий политику и `log = {"policies": [...], "success": [...], "return": [...]}`. Можно скопировать своё решение с семинара.

Одно отличие от семинарской версии: добавьте циклу параметры `gamma` и `max_steps` и передавайте их дальше в `discounted_return` и `run_session`. В частях 3 и 4 они понадобятся: там тот же алгоритм учится на складе, где считается прибыль за 30 дней без дисконтирования.

Обучите агента на `env8`, постройте кривую обучения (доля успехов и средний return по итерациям) и нарисуйте выученную политику. Самопроверка требует **долю успехов ≥ 90 %** у выученной стохастической политики. Подсказка: успешных эпизодов у случайной политики очень мало, поэтому сессий нужно несколько сотен на итерацию; помогает сглаживание.

In [ ]:
def select_elite(sessions, scores, q=0.7):
    # TODO: ваш код здесь
    raise NotImplementedError

def update_policy(policy, elite, laplace=0.0, mix=1.0):
    # TODO: ваш код здесь
    raise NotImplementedError

def cross_entropy_method(env, n_iter=15, n_sessions=200, q=0.7, laplace=0.0, mix=1.0,
                         seed=0, gamma=GAMMA, max_steps=100):
    # TODO: ваш код здесь
    raise NotImplementedError

In [ ]:
# TODO: подберите n_iter, n_sessions, q, laplace, mix
policy8, log8 = cross_entropy_method(env8, n_iter=..., n_sessions=..., q=..., laplace=..., mix=..., seed=0)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(log8["success"], marker="."); axes[0].set_xlabel("итерация"); axes[0].set_ylabel("доля успешных эпизодов")
axes[1].plot(log8["return"], marker=".", color="C1"); axes[1].set_xlabel("итерация"); axes[1].set_ylabel("средний return")
draw_policy(policy8, desc8, ax=axes[2], title="выученная политика"); plt.tight_layout(); plt.show()

rate8 = success_rate(env8, policy8, n_episodes=1000)
print(f"доля успехов: {rate8:.1%}")
assert policy8.shape == (64, 4) and np.allclose(policy8.sum(axis=1), 1)
assert rate8 >= 0.9, "политика должна доходить до цели не реже, чем в 90% эпизодов"

### 1.2 Сглаживание (5 баллов)

Сравните три варианта обновления политики: **без сглаживания** (`laplace=0, mix=1`), **только Лаплас** (`laplace=0.5, mix=1`), **Лаплас и смешивание** (`laplace=0.5, mix=0.5`). Для каждого варианта сделайте 3 запуска с разными сидами и постройте кривые доли успехов (все девять на одном графике, цвет — вариант). Ответьте текстом: какой вариант быстрее, какой стабильнее и у какого чаще встречаются запуски, «зависшие» на нуле?

In [ ]:
# TODO: ваш код здесь (9 запусков, один график)

_Ваш ответ:_ TODO

### 1.3 Скользкий лёд (5 баллов)

Вернёмся на озеро 4×4, но включим скольжение: `gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)`. Запустите `cross_entropy_method` с параметрами из 1.1 и постройте кривую. Затем:

1. Объясните текстом, почему результат намного хуже, чем на гладком льду: что именно попадает в элиту и почему это плохо для шага 3?
2. Предложите **не меньше двух** изменений (больше эпизодов на итерацию, выше `q`, меньше `mix`, больше итераций) и проверьте каждое на графике. Цель — доля успехов ≥ 35 %. Лучшая детерминированная таблица даёт около 74 %, табличный Cross-Entropy до неё не дотягивает, и это нормально — объясните, почему.

In [ ]:
slip = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
# TODO: ваш код здесь

_Ваш ответ:_ TODO

### 1.4 Два вопроса про выученную политику (3 балла)

1. Стрелки на картинке политики показывают самое вероятное действие. Почему **жадная** версия выученной политики (`argmax` в каждой клетке) иногда не доходит до цели, хотя стохастическая доходит почти всегда? Подсказка: посмотрите на клетки с точкой.
2. Почему нулевая вероятность действия опаснее, чем просто маленькая? Свяжите ответ с разделом про исследование и использование.

_Ваш ответ:_ TODO

## Часть 2. Оценка политики по Монте-Карло (10 баллов)

### 2.1 Три политики (5 баллов)

Оцените методом Монте-Карло три политики на `env8`: равномерную, **маршрут, написанный руками** (словарь «клетка → действие», как на семинаре; подсказка: вдоль верхнего края, потом вдоль правого) и выученную в 1.1. Для каждой посчитайте средний return и его стандартную ошибку по 1000 эпизодам и нарисуйте столбчатую диаграмму с ошибками.

Функцию `evaluate` сделайте с параметрами `gamma` и `max_steps` и возвращайте пару «среднее, стандартная ошибка». Это единственная функция оценки на весь ноутбук: в частях 3 и 4 ей же оцениваются политики склада.

Ответьте текстом: сколько эпизодов нужно, чтобы стандартная ошибка оценки равномерной политики стала меньше 0.001?

In [ ]:
LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3

def table_policy(route, n_states=64, n_actions=4, default=LEFT):
    policy = np.zeros((n_states, n_actions))
    for s in range(n_states):
        policy[s, route.get(s, default)] = 1.0
    return policy

def evaluate(env, policy, n_episodes=1000, seed=0, gamma=GAMMA, max_steps=1000):
    # TODO: вернуть (средний return, стандартная ошибка)
    raise NotImplementedError

route8 = {}   # TODO: маршрут руками
uniform8 = np.ones((64, 4)) / 4
# TODO: оценка трёх политик и диаграмма

_Ваш ответ:_ TODO

### 2.2 Тепловая карта ценности (5 баллов)

Возьмите `estimate_values(sessions, n_states)` с семинара (ценность клетки — средний return от первого попадания в неё) и постройте тепловую карту $V^\pi$ для выученной политики из 1.1 по 3000 эпизодам (`plot_values` есть в служебной ячейке). Ответьте текстом: почему ценность растёт к цели и чему равен её максимум; что означают прочерки; почему у прорубей ценности нет.

In [ ]:
def estimate_values(sessions, n_states, gamma=GAMMA):
    # TODO: ваш код здесь
    raise NotImplementedError

# TODO: тепловая карта V для policy8

_Ваш ответ:_ TODO

## Часть 3. Своя среда: управление запасами (20 баллов)

До сих пор среду давали готовой. Теперь напишем свою — и убедимся, что функциям из частей 1 и 2 всё равно, чья это среда.

**Задача.** Вы управляете складом одного товара. Каждый день:

1. утром на складе `stock` единиц (от 0 до `capacity`);
2. вы решаете, сколько **заказать**: действие $a \in \{0, 1, \ldots, \text{max\_order}\}$; заказ приезжает сразу, но склад не вмещает больше `capacity` — лишнее пропадает;
3. днём приходит **случайный спрос** $d$ (Пуассон со средним `demand_mean`), продаётся $\min(d, \text{stock})$ единиц;
4. вечером остаток переходит на завтра. Эпизод длится `horizon` дней.

Экономика дня: выручка `price` за проданную единицу, `order_cost` за **каждую заказанную** (даже если она не поместилась), `holding_cost` за каждую единицу, оставшуюся к вечеру, и `stockout_cost` за каждую единицу неудовлетворённого спроса.

Это классическая задача исследования операций, и RL здесь не единственный способ, зато прекрасный полигон: дискретное состояние, дискретное действие, честная случайность — и, главное для лекции 2, распределение спроса известно. Значит, в части 4 модель $P$ и $R$ можно будет выписать **точно** и решить уравнения Беллмана, как на лекции.

### 3.1 Класс среды (12 баллов)

Реализуйте `InventoryEnv`:

* `observation_space = Discrete(capacity + 1)` — наблюдение это текущий остаток;
* `action_space = Discrete(max_order + 1)`;
* `reset(seed)` — остаток `initial_stock`, день 0, вернуть `(obs, info)`;
* `step(action)` — по описанию выше; в `info` положите `demand` и `sold`;
* спрос генерируйте через `self.np_random.poisson(self.demand_mean)` — тогда среда воспроизводима по сиду.

Подумайте, какой из флагов, `terminated` или `truncated`, правильно выставлять в конце горизонта, и объясните выбор в комментарии. Подсказка: после `horizon` дней задача **закончена по условию**, а не оборвана снаружи.

In [ ]:
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

In [ ]:
class InventoryEnv(gym.Env):
    metadata = {"render_modes": ["ansi"]}

    def __init__(self, capacity=20, max_order=10, demand_mean=4.0, horizon=30, initial_stock=10,
                 price=5.0, order_cost=2.0, holding_cost=0.5, stockout_cost=3.0, render_mode=None):
        self.capacity, self.max_order = capacity, max_order
        self.demand_mean, self.horizon, self.initial_stock = demand_mean, horizon, initial_stock
        self.price, self.order_cost = price, order_cost
        self.holding_cost, self.stockout_cost = holding_cost, stockout_cost
        self.render_mode = render_mode

        # TODO: observation_space и action_space
        self.observation_space = None
        self.action_space = None

        self.stock = initial_stock
        self.day = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # TODO: вернуть (наблюдение, info) и сбросить день
        raise NotImplementedError

    def step(self, action):
        # TODO: заказ -> спрос -> продажа -> награда -> конец горизонта
        raise NotImplementedError

    def render(self):
        if self.render_mode == "ansi":
            return f"день {self.day:2d}: на складе {self.stock:2d}"

In [ ]:
# Самопроверка 3.1
env = InventoryEnv()
check_env(env)

obs, info = env.reset(seed=0)
assert obs == 10 and env.observation_space.contains(obs)
assert env.action_space.n == 11 and env.observation_space.n == 21

# заказ выше вместимости обрезается: 10 + 10 = 20, больше capacity не бывает
env.reset(seed=0)
obs, r, term, trunc, info = env.step(10)
assert 0 <= obs <= 20 and "demand" in info and "sold" in info
assert info["sold"] <= 20 and obs == 20 - info["sold"], "продано не может быть больше остатка после заказа"

# эпизод длится ровно horizon дней
env.reset(seed=1)
for day in range(30):
    obs, r, term, trunc, info = env.step(0)
    assert term == (day == 29), "terminated должен стать True ровно после horizon-го дня"

# воспроизводимость: одинаковый seed -> одинаковый спрос
def demands(seed):
    e = InventoryEnv(); e.reset(seed=seed)
    return [e.step(3)[4]["demand"] for _ in range(10)]
assert demands(42) == demands(42) and demands(42) != demands(43)
print("OK: InventoryEnv проходит проверки")

### 3.2 Ручные политики (8 баллов)

Политика здесь — такая же таблица, как на озере: строка на каждый остаток, столбец на каждое количество заказа. Правило «остаток → заказ» превращается в таблицу одной функцией `rule_to_table`, после чего работают и `run_session`, и `evaluate` из части 2.

Сравните три ручные политики:

1. **ничего не заказывать**;
2. **заказывать фиксированно** `k` единиц каждый день — подберите `k` перебором от 0 до 10;
3. **пополнять до уровня** $S$: заказать $\max(0, S - \text{stock})$, но не больше `max_order` (классическая base-stock policy) — подберите $S$.

Здесь сравниваем **прибыль за 30 дней**, все дни равноценны: вызывайте `evaluate(..., gamma=1.0, max_steps=31)`. Постройте график «средняя прибыль в зависимости от параметра» для политик 2 и 3. Какая из ручных политик лучшая и почему это ожидаемо? Лучшую base-stock политику сохраните таблицей `pi_base` — она понадобится в части 4.

In [ ]:
def rule_to_table(rule, n_states=21, n_actions=11):
    """Правило «остаток -> сколько заказать» превращаем в таблицу-политику."""
    # TODO: ваш код здесь
    raise NotImplementedError

# TODO: три политики, перебор параметров, график
pi_base = ...   # TODO: таблица лучшей base-stock политики

In [ ]:
# Самопроверка 3.2
_m, _se = evaluate(InventoryEnv(), rule_to_table(lambda s: 0), n_episodes=20, gamma=1.0, max_steps=31)
assert np.isscalar(_m) and np.isscalar(_se) and _se >= 0, "evaluate должна вернуть пару чисел: среднее и стандартную ошибку"
assert pi_base.shape == (21, 11) and np.allclose(pi_base.sum(axis=1), 1), "pi_base — таблица 21 × 11, строки суммируются в единицу"
print(f"OK: прибыль политики «ничего не заказывать» = {_m:.1f} ± {_se:.1f}")

_Ваш ответ:_ TODO

## Часть 4. Уравнения Беллмана на складе (25 баллов)

### 4.1 Точная модель MDP (8 баллов)

Спрос — Пуассон с известным средним, значит, модель переходов можно выписать явно. Реализуйте `inventory_mdp(env)`, возвращающую `P[s, a, s']` и `R[s, a]` для `InventoryEnv` — ровно тот формат, что на лекции:

* после заказа на складе $m = \min(s + a, \text{capacity})$ единиц;
* при спросе $d$ продаётся $\min(d, m)$, остаток $s' = \max(m - d, 0)$; все значения спроса $d \ge m$ приводят в $s' = 0$ — не забудьте хвост распределения (функция `poisson_pmf` ниже уже это делает);
* $R[s, a]$ — **ожидаемая** награда дня: $\mathbb E_d[\,\text{price}\cdot\min(d, m) - \text{order\_cost}\cdot a - \text{holding\_cost}\cdot\max(m - d, 0) - \text{stockout\_cost}\cdot\max(d - m, 0)\,]$; хвост можно обрезать на $d_{\max} = 60$;
* горизонт в этой части считаем бесконечным, а задачу — дисконтированной с $\gamma = 0.95$ (`GAMMA` уже определена): тогда ценность не зависит от дня, и уравнения из лекции применимы дословно. Как учесть конечный горизонт — вопрос 5.4.

In [ ]:
def poisson_pmf(mean, d_max=60):
    """Вероятности спроса d = 0..d_max для Пуассона; хвост d > d_max добавлен к последней точке."""
    pmf = np.zeros(d_max + 1)
    pmf[0] = np.exp(-mean)
    for d in range(1, d_max + 1):
        pmf[d] = pmf[d - 1] * mean / d
    pmf[-1] += 1 - pmf.sum()
    return pmf

def inventory_mdp(env, d_max=60):
    """P[s, a, s'] и R[s, a] для InventoryEnv при бесконечном горизонте."""
    n_s, n_a = env.capacity + 1, env.max_order + 1
    P, R = np.zeros((n_s, n_a, n_s)), np.zeros((n_s, n_a))
    pmf = poisson_pmf(env.demand_mean, d_max)
    # TODO: для каждого s, a: m = min(s + a, capacity); для каждого d: s' = max(m - d, 0), награда дня,
    #       P[s, a, s'] += pmf[d], R[s, a] += pmf[d] * награда
    raise NotImplementedError

In [ ]:
# Самопроверка 4.1
env = InventoryEnv()
P, R = inventory_mdp(env)
assert P.shape == (21, 11, 21) and R.shape == (21, 11)
assert np.allclose(P.sum(axis=2), 1.0), "строки P — распределения"
assert np.all(P[:, :, 0] > 0), "нулевой остаток достижим из любого состояния (большой спрос)"
assert abs(P[20, 5, 20] - np.exp(-env.demand_mean)) < 1e-12, "полный склад остаётся полным только при нулевом спросе"
# ожидаемая награда при s=10, a=0 сходится с Монте-Карло по среде
_e = InventoryEnv(); _rs = []
for _seed in range(3000):
    _e.reset(seed=_seed); _rs.append(_e.step(0)[1])
assert abs(np.mean(_rs) - R[10, 0]) < 0.3, f"R[10, 0] = {R[10, 0]:.2f}, а Монте-Карло даёт {np.mean(_rs):.2f}"
print("OK: модель MDP склада построена")

### 4.2 Ценность ручной политики: точно и по Монте-Карло (7 баллов)

Посчитайте $V^\pi$ для `pi_base` из 3.2 двумя способами:

* **точно**, как решение линейной системы $(I - \gamma P_\pi) V^\pi = r_\pi$ — уравнение Беллмана для заданной стратегии без максимума линейно, это было на лекции;
* **по Монте-Карло** — функцией `evaluate` из части 2, по 300 эпизодов из каждого стартового остатка: `evaluate(InventoryEnv(initial_stock=s, horizon=150), pi_base, n_episodes=300, gamma=GAMMA, max_steps=150)`. Длинный горизонт имитирует бесконечный: $0.95^{150} \approx 5 \cdot 10^{-4}$.

Постройте график «остаток → $V^\pi$» с обеими кривыми. Объясните, почему они должны совпасть и откуда берутся расхождения.

In [ ]:
def policy_evaluation_exact(P, R, policy, gamma=GAMMA):
    # TODO: V^π = (I - γ P_π)^{-1} r_π, где P_π[s, s'] = Σ_a π[s, a] P[s, a, s'], r_π[s] = Σ_a π[s, a] R[s, a]
    raise NotImplementedError

# TODO: точная V^π, Монте-Карло V^π по стартовым остаткам, график

In [ ]:
# Самопроверка 4.2
V_base = policy_evaluation_exact(P, R, pi_base)
assert V_base.shape == (21,) and np.all(np.diff(V_base) > -1e-9), "ценность не убывает с ростом остатка: товар на складе не бывает вреден при разумной политике"
print(f"OK: V^π(остаток 10) = {V_base[10]:.1f}")

_Ваш ответ:_ TODO

### 4.3 Оптимальная политика заказов (10 баллов)

Реализуйте value iteration и найдите $V^*$ и $Q^*$ для склада. Постройте оптимальную политику $\pi^*(s) = \arg\max_a Q^*(s, a)$ и изобразите её графиком «остаток → заказ» рядом с `pi_base`. Затем обучите на складе свой `cross_entropy_method` из части 1 (`gamma=1.0, max_steps=31`, квантиль около 0.6, `laplace=1, mix=0.5`, 30 итераций по 150 эпизодов) и оцените все три политики **одной и той же** функцией `evaluate` на длинной среде `InventoryEnv(horizon=200)` с `gamma=GAMMA`.

Ответьте текстом:

* похожа ли $\pi^*$ на «пополнять до уровня $S$» и совпала ли она с лучшей ручной политикой? Если да — это не случайность: для такой модели склада оптимальность политики вида base-stock — классический результат теории запасов, и уравнение оптимальности его переоткрыло. Объясните своими словами, почему заказывать до фиксированного уровня разумно;
* сколько эпизодов понадобилось value iteration и сколько — Cross-Entropy? Почему Cross-Entropy с таблицей $21 \times 11$ проигрывает, хотя видел несколько тысяч эпизодов?

In [ ]:
def value_iteration(P, R, gamma=GAMMA, tol=1e-8):
    # TODO: V ← max_a [R[s, a] + γ Σ_s' P[s, a, s'] V[s']] до сходимости; вернуть V*, Q*
    raise NotImplementedError

# TODO: V*, Q*, π*, график «остаток → заказ», Cross-Entropy, честное сравнение трёх политик

In [ ]:
# Самопроверка 4.3
V_star, Q_star = value_iteration(P, R)
pi_star = np.eye(11)[Q_star.argmax(axis=1)]
assert np.allclose(V_star, Q_star.max(axis=1))
assert np.all(V_star >= V_base - 1e-6), "V* не меньше ценности ручной политики во всех состояниях"
assert np.allclose(policy_evaluation_exact(P, R, pi_star), V_star, atol=1e-4), "жадная по Q* политика имеет ценность V*"
print(f"OK: V*(10) = {V_star[10]:.1f}, у ручной политики V^π(10) = {V_base[10]:.1f}")

_Ваш ответ:_ TODO

## Часть 5. Теория (20 баллов)

Отвечайте прямо в markdown-ячейках, используя LaTeX ($...$ или $$...$$). Показывайте промежуточные шаги, а не только финальный ответ.

### 5.1 Марковская цепь (3 балла)

Погода в городе описывается марковской цепью с состояниями {Солнце, Облачно, Дождь} и матрицей переходов

$$
P = \begin{pmatrix} 0.7 & 0.2 & 0.1 \\ 0.3 & 0.4 & 0.3 \\ 0.2 & 0.3 & 0.5 \end{pmatrix}.
$$

1. Сегодня солнце. Какова вероятность дождя послезавтра? Запишите вычисление через $p_0 P^2$.
2. Найдите стационарное распределение $\pi^\top P = \pi^\top$ — аналитически или численно в ячейке ниже, но систему уравнений запишите.
3. Проверьте численно: возведите $P$ в большую степень и сравните строки с найденным $\pi$.

_Ваш ответ:_ TODO

In [ ]:
P_weather = np.array([[0.7, 0.2, 0.1],
                      [0.3, 0.4, 0.3],
                      [0.2, 0.3, 0.5]])
# TODO: p_0 @ P^2, стационарное распределение, проверка через np.linalg.matrix_power

### 5.2 Уравнение Беллмана и связь V с Q (6 баллов)

1. Исходя из определения $V^\pi(s) = \mathbb E_\pi[G_t \mid S_t = s]$ и тождества $G_t = R_t + \gamma G_{t+1}$, выведите уравнение Беллмана для $V^\pi$. Явно укажите, где используется марковское свойство и где — то, что политика не зависит от времени.
2. Докажите две связи: $V^\pi(s) = \sum_a \pi(a \mid s)\, Q^\pi(s, a)$ и $Q^\pi(s, a) = r(s, a) + \gamma \sum_{s'} P(s' \mid s, a)\, V^\pi(s')$. Выведите из них, что $V^*(s) = \max_a Q^*(s, a)$, и объясните, почему максимум появляется именно здесь, а не во второй связи.

### 5.3 Ценность руками (4 балла)

Цепочка $s_0 \to s_1 \to s_2$, политика и переходы детерминированные (всегда «вправо»), награда $+1$ за приход в $s_2$, $s_2$ терминальное, $\gamma = 0.9$. Выпишите уравнения Беллмана для $V^\pi(s_0)$ и $V^\pi(s_1)$ и решите их. Затем сделайте переход $s_1 \to s_2$ случайным: с вероятностью $0.5$ агент остаётся в $s_1$. Решите заново и объясните, почему $V^\pi(s_1)$ стало меньше единицы.

### 5.4 Склад как MDP (4 балла)

Опишите `InventoryEnv` как MDP $\langle \mathcal S, \mathcal A, P, R, \gamma \rangle$: множества, формула $P(s' \mid s, a)$ через распределение спроса (с обрезанием по `capacity` и хвостом в $s' = 0$) и $r(s, a)$. В самой среде эпизод длится `horizon` дней и заканчивается с `terminated=True`. Что это означает для уравнения Беллмана: почему ценность последнего дня равна ожидаемой награде за день и как выглядит уравнение для $V_t(s)$, когда ценность зависит от номера дня? Как включить номер дня в состояние, чтобы вернуться к стационарной постановке из лекции?

### 5.5 Лотерея Колобка: обобщение (3 балла)

На лекции решено уравнение оптимальности для лотереи с билетом за 10 и выигрышем 1000 при $\gamma = 1$. Решите его для произвольной цены билета $c$ и выигрыша $W$: найдите порог $p^*$, при котором «купить» и «не покупать» равноценны, и проверьте, что при $c = 10$, $W = 1000$ получается $0.99$. Как сдвинется порог при $\gamma < 1$ и почему?

_Ваши ответы на 5.2–5.5:_ TODO

## Бонус А. Скрытый день недели (10 баллов)

Иллюстрация марковского свойства из лекции 2. Пусть спрос зависит от дня недели: в будни средний спрос `demand_mean`, в субботу и воскресенье — в `weekend_factor` раз больше (по умолчанию втрое: в выходные спрос выше, чем можно заказать за день, и выгодно запасаться заранее). День недели — это `day % 7`, день 0 — понедельник.

Реализуйте `WeekdayDemandEnv(InventoryEnv)` с параметром `observe_weekday`:

* `observe_weekday=False`: наблюдение — только остаток, как раньше (`Discrete(capacity + 1)`). Агент не видит, что завтра выходные, — марковское свойство нарушено *нашим выбором наблюдения*;
* `observe_weekday=True`: наблюдение — пара «остаток, день недели», закодированная одним числом `stock * 7 + weekday` в `Discrete((capacity + 1) * 7)`.

Спрос дня разыгрывайте по **текущему** дню и только потом увеличивайте счётчик дней.

Сначала сравните **ручные** политики: обычную «пополнять до уровня $S$» и её версию с двумя уровнями — $S$ в будни и $S_w$ перед выходными (в пятницу и субботу). Затем обучите свой Cross-Entropy в обоих вариантах наблюдения (одинаковые гиперпараметры, 30 итераций, несколько сидов). Оценивайте всё **одной** функцией `evaluate` на среде с `observe_weekday=True`; политику без дня недели разверните в таблицу $147 \times 11$ через `np.repeat(policy, 7, axis=0)` — строка `stock * 7 + weekday` получит строку `stock`.

Ответьте текстом: почему ручная политика с двумя уровнями выигрывает у одного уровня; во сколько раз выросла таблица политики, когда день недели стал виден, и выиграл ли от этой информации Cross-Entropy — если нет, то почему. Что будет при `weekend_factor = 1.0`?

In [ ]:
class WeekdayDemandEnv(InventoryEnv):
    def __init__(self, observe_weekday=False, weekend_factor=3.0, **kwargs):
        super().__init__(**kwargs)
        self.observe_weekday, self.weekend_factor = observe_weekday, weekend_factor
        # TODO: observation_space в зависимости от observe_weekday

    def _obs(self):
        # TODO: остаток или stock * 7 + weekday
        raise NotImplementedError

    def _demand_mean_today(self):
        # TODO: demand_mean * weekend_factor в субботу и воскресенье (self.day % 7 in (5, 6)), иначе demand_mean
        raise NotImplementedError

    def reset(self, seed=None, options=None):
        # TODO: как в InventoryEnv, но вернуть self._obs()
        raise NotImplementedError

    def step(self, action):
        # TODO: как в InventoryEnv, но спрос ~ Poisson(self._demand_mean_today()) и наблюдение self._obs()
        raise NotImplementedError

# TODO: ручные политики, Cross-Entropy в двух вариантах, честная оценка, сравнение

In [ ]:
# Самопроверка бонуса А
for flag, n_obs in [(False, 21), (True, 21 * 7)]:
    _e = WeekdayDemandEnv(observe_weekday=flag)
    check_env(_e)
    assert _e.observation_space.n == n_obs, f"observe_weekday={flag}: ожидали Discrete({n_obs})"
_e = WeekdayDemandEnv(observe_weekday=True)
obs, _ = _e.reset(seed=0)
assert obs == 10 * 7 + 0, "в день 0 (понедельник) при остатке 10 наблюдение должно быть 70"
# в выходные спрос в среднем выше: усредняем по многим сидам
def mean_demand(day_index, n=300):
    total = 0.0
    for s in range(n):
        e = WeekdayDemandEnv(); e.reset(seed=s)
        for d in range(day_index + 1):
            info = e.step(10)[4]
        total += info["demand"]
    return total / n
assert mean_demand(5) > 1.5 * mean_demand(2), "в субботу спрос должен быть заметно выше, чем в среду"
print("OK: WeekdayDemandEnv проходит проверки")

_Ваш ответ:_ TODO

## Бонус Б. Многорукие бандиты (10 баллов)

**Многорукий бандит** — MDP с одним состоянием: есть $K$ игровых автоматов («рук»), у каждой своя неизвестная вероятность выигрыша, на каждом шаге вы дёргаете одну руку и получаете 0 или 1. Переходов между состояниями нет, зато дилемма исследования и использования — в чистом виде. **Ценность действия** здесь просто средний выигрыш руки, $Q(a) = \mathbb{E}[R \mid A = a]$: та же $Q$, что в лекции 2, только без состояний.

Среда дана. Реализуйте агента **ε-greedy**: с вероятностью ε он выбирает случайную руку, иначе руку с наибольшей оценкой $\hat Q(a)$; оценка обновляется инкрементально после каждого шага, $\hat Q(a) \leftarrow \hat Q(a) + \frac{1}{N(a)}\big(r - \hat Q(a)\big)$ — это то же самое, что среднее по всем наградам руки. Постройте кривые накопленного **regret** $\sum_t (p^* - p_{A_t})$ для ε ∈ {0, 0.01, 0.1}, усреднив по 20 запускам. Объясните форму кривых: почему у ε = 0 regret иногда растёт быстрее всех, а у ε = 0.1 — линейно даже после того, как лучшая рука найдена.

Из 10 баллов 7 — за ε-greedy с кривыми и объяснением, ещё 3 — за UCB1 ($A_t = \arg\max_a \hat Q(a) + c\sqrt{\ln t / N(a)}$): реализуйте его и добавьте на график.

In [ ]:
class BernoulliBanditEnv:
    def __init__(self, probs, rng):
        self.probs, self.rng = np.array(probs, dtype=float), rng
    def pull(self, arm):
        return float(self.rng.random() < self.probs[arm])

class EpsilonGreedyAgent:
    def __init__(self, n_arms, epsilon, rng):
        self.epsilon, self.rng = epsilon, rng
        self.Q, self.N = np.zeros(n_arms), np.zeros(n_arms)
    def select_arm(self):
        # TODO: ваш код здесь
        raise NotImplementedError
    def update(self, arm, reward):
        # TODO: ваш код здесь
        raise NotImplementedError

def run_bandit(make_agent, probs=(0.1, 0.5, 0.3, 0.55, 0.45), n_steps=2000, n_runs=20):
    # средний накопленный regret по n_runs запускам
    regrets = []
    for run in range(n_runs):
        rng = np.random.default_rng(run)
        env, agent = BernoulliBanditEnv(probs, rng), make_agent(len(probs), rng)
        regret, total = [], 0.0
        for t in range(n_steps):
            arm = agent.select_arm(); r = env.pull(arm); agent.update(arm, r)
            total += env.probs.max() - env.probs[arm]; regret.append(total)
        regrets.append(regret)
    return np.mean(regrets, axis=0)

# TODO: кривые regret для ε ∈ {0, 0.01, 0.1}

_Ваш ответ:_ TODO

## Бонус В. DL-разминка (10 баллов)

Три упражнения из `../../01-intro/seminar/dl_basics.ipynb`, за них 3, 3 и 4 балла. Нужен PyTorch (`pip install torch`, хватит версии для процессора).

### В.1 Градиент руками и autograd (3 балла)

Для линейной модели $\hat y = w x + b$ и функции потерь $L = \frac{1}{n}\sum_i (\hat y_i - y_i)^2$ выпишите $\partial L / \partial w$ и $\partial L / \partial b$ и посчитайте их в numpy. Сравните с тем, что даёт `torch.autograd`.

In [ ]:
import torch

rng = np.random.default_rng(0)
x = rng.uniform(-1, 1, 50); y = 2 * x - 1 + rng.normal(0, 0.1, 50)
w0, b0 = 0.5, 0.0

def grad_manual(w, b, x, y):
    # TODO: вернуть (dL/dw, dL/db) по формулам
    raise NotImplementedError

w = torch.tensor(w0, requires_grad=True); b = torch.tensor(b0, requires_grad=True)
loss = ((w * torch.tensor(x) + b - torch.tensor(y)) ** 2).mean()
loss.backward()
dw, db = grad_manual(w0, b0, x, y)
assert np.isclose(dw, w.grad.item(), atol=1e-6) and np.isclose(db, b.grad.item(), atol=1e-6)
print("ok: градиенты совпали", round(dw, 4), round(db, 4))

### В.2 Регрессия нейросетью (3 балла)

Обучите небольшую сеть (`nn.Sequential` из двух скрытых слоёв по 32 нейрона с `Tanh`) приближать $f(x) = \sin 3x$ на отрезке $[-1, 1]$. Цель — среднеквадратичная ошибка на сетке из 200 точек меньше 0.01. Нарисуйте функцию и её приближение.

In [ ]:
import torch.nn as nn

xs = torch.linspace(-1, 1, 200).unsqueeze(1)
ys = torch.sin(3 * xs)

# TODO: модель, оптимизатор, цикл обучения
model = ...

with torch.no_grad():
    mse = ((model(xs) - ys) ** 2).mean().item()
plt.plot(xs, ys, label="sin 3x"); plt.plot(xs, model(xs).detach(), "--", label="сеть"); plt.legend(); plt.show()
print(f"MSE = {mse:.4f}")
assert mse < 0.01

### В.3 Классификация: две спирали (4 балла)

Данные — две перепутанные спирали (генератор ниже). Обучите классификатор (`nn.Sequential`, на выходе два логита, функция потерь `nn.CrossEntropyLoss` — та же кросс-энтропия, что в методе Cross-Entropy из лекции) и добейтесь точности не ниже 95 % на отложенной выборке. Нарисуйте точки и раскраску плоскости предсказаниями сети.

In [ ]:
def make_spirals(n=300, noise=0.15, seed=0):
    rng = np.random.default_rng(seed)
    t = np.linspace(0.4, 3.0, n)
    X, y = [], []
    for k in range(2):
        angle = t * np.pi + k * np.pi
        X.append(np.stack([t * np.cos(angle), t * np.sin(angle)], 1) + rng.normal(0, noise, (n, 2))); y.append(np.full(n, k))
    X, y = np.concatenate(X).astype(np.float32), np.concatenate(y)
    idx = rng.permutation(len(y))
    return X[idx], y[idx]

X, y = make_spirals()
X_train, y_train, X_test, y_test = torch.tensor(X[:400]), torch.tensor(y[:400]), torch.tensor(X[400:]), torch.tensor(y[400:])

# TODO: модель, обучение
clf = ...

with torch.no_grad():
    acc = (clf(X_test).argmax(1) == y_test).float().mean().item()
print(f"точность на отложенной выборке: {acc:.1%}")
assert acc >= 0.95

## Чек-лист перед сдачей

- [ ] 1.1: `cross_entropy_method` (с параметрами `gamma` и `max_steps`) проходит самопроверку на 8×8, есть кривые и картинка политики
- [ ] 1.2: девять кривых на одном графике и вывод про сглаживание
- [ ] 1.3: кривые на скользком льду, объяснение и не меньше двух проверенных изменений
- [ ] 1.4: ответы на два вопроса
- [ ] 2.1: три политики оценены со стандартными ошибками, есть диаграмма и ответ про число эпизодов
- [ ] 2.2: тепловая карта ценности и её объяснение
- [ ] 3.1: `InventoryEnv` проходит `check_env` и самопроверку
- [ ] 3.2: ручные политики оценены, есть график по параметру и выбрана лучшая
- [ ] 4.1: `inventory_mdp` проходит самопроверку
- [ ] 4.2: точная ценность ручной политики совпадает с Монте-Карло в пределах ошибки, есть график
- [ ] 4.3: `value_iteration` проходит самопроверку, есть сравнение оптимальной, ручной и выученной Cross-Entropy политик
- [ ] 5.1–5.5: марковская цепь, вывод уравнения Беллмана, ценность руками, склад как MDP, порог лотереи
- [ ] Коды квизов по лекциям 1 и 2 лежат в `quiz/week01.txt` и `quiz/week02.txt`
- [ ] Ноутбук выполнен целиком: «Restart & Run All» перед сдачей
- [ ] (бонус) А: `WeekdayDemandEnv` проходит самопроверку, есть сравнение и ответ
- [ ] (бонус) Б: ε-greedy и кривые regret, по желанию UCB1
- [ ] (бонус) В: три упражнения DL-разминки проходят `assert`